# EquiLend AI — Phase 1: HMEQ Data Exploration

**Dataset:** HMEQ (Home Equity) — 5,960 home equity loan records  
**Target:** `BAD` (1 = defaulted, 0 = repaid)

Checklist:
- [ ] Load and inspect raw data
- [ ] Check shape, dtypes, missing values
- [ ] Class balance (default vs repaid)
- [ ] Feature distributions by outcome
- [ ] Correlation matrix
- [ ] Summary stats & next steps

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.insert(0, '..')

from src.data.schema import (
    FEATURE_SCHEMA, TARGET_COLUMN, MODEL_FEATURES,
    NUMERIC_FEATURES, CATEGORICAL_FEATURES, NULLABLE_FEATURES,
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Load Data

Drop your CSV into `data/raw/` and update the path below.

In [ ]:
DATA_PATH = '../data/raw/hmeq.csv'

df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

## 2. Data Quality

In [ ]:
print('=== Dtypes ===')
print(df.dtypes)

print('\n=== Missing Values ===')
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.concat([missing, missing_pct], axis=1, keys=['count', '%'])
print(missing_df[missing_df['count'] > 0])

print(f'\nTotal rows: {len(df)}')
print(f'Complete rows (no nulls): {df.dropna().shape[0]} ({df.dropna().shape[0]/len(df)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(10, 4))
missing_pct[missing_pct > 0].plot(kind='barh', color='#e74c3c', ax=ax)
ax.set_xlabel('% Missing')
ax.set_title('Missing Values by Feature')
plt.tight_layout()
plt.show()

## 3. Class Balance (BAD: 1=Default, 0=Repaid)

In [ ]:
counts = df[TARGET_COLUMN].value_counts()
print(counts)
print(f'\nDefault rate: {df[TARGET_COLUMN].mean()*100:.1f}%')

fig, ax = plt.subplots(figsize=(6, 4))
counts.plot(kind='bar', color=['#2ecc71', '#e74c3c'], ax=ax)
ax.set_xticklabels(['Repaid (0)', 'Default (1)'], rotation=0)
ax.set_ylabel('Count')
ax.set_title(f'Class Balance — {TARGET_COLUMN}')
for i, v in enumerate(counts):
    ax.text(i, v + 30, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Numeric Feature Distributions

In [ ]:
n = len(NUMERIC_FEATURES)
fig, axes = plt.subplots((n + 2) // 3, 3, figsize=(18, (n + 2) // 3 * 4))
axes = axes.flatten()

for i, col in enumerate(NUMERIC_FEATURES):
    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        subset = df[df[TARGET_COLUMN] == label][col].dropna()
        axes[i].hist(subset, bins=30, alpha=0.6, color=color,
                     label='Repaid' if label == 0 else 'Default')
    axes[i].set_title(col, fontsize=11)
    axes[i].legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric Feature Distributions by Outcome', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

## 5. Correlation Matrix (model features only)

In [ ]:
corr = df[NUMERIC_FEATURES].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, square=True)
plt.title('Feature Correlation Matrix (Numeric Features)')
plt.tight_layout()
plt.show()

## 6. Categorical Feature Breakdown

In [ ]:
fig, axes = plt.subplots(1, len(CATEGORICAL_FEATURES), figsize=(14, 5))
if len(CATEGORICAL_FEATURES) == 1:
    axes = [axes]

for ax, col in zip(axes, CATEGORICAL_FEATURES):
    ct = pd.crosstab(df[col].fillna('Missing'), df[TARGET_COLUMN], normalize='index')
    ct.columns = ['Repaid', 'Default']
    ct.plot(kind='bar', stacked=True, ax=ax, color=['#2ecc71', '#e74c3c'])
    ax.set_title(f'Default Rate by {col}')
    ax.set_ylabel('Proportion')
    ax.set_xlabel('')
    ax.legend(fontsize=8)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()

## 7. Summary Statistics

In [ ]:
print('=== Summary Statistics ===')
display(df.describe().T.round(2))

print('\n=== Key Takeaways ===')
print(f'  Records:       {len(df):,}')
print(f'  Features:      {len(MODEL_FEATURES)} ({len(NUMERIC_FEATURES)} numeric, {len(CATEGORICAL_FEATURES)} categorical)')
print(f'  Default rate:  {df[TARGET_COLUMN].mean()*100:.1f}%')
print(f'  Missing cols:  {df.isnull().any().sum()} of {len(df.columns)}')
print(f'  Imputation:    median (numeric), mode (categorical)')
print(f'  scale_pos_weight for XGBoost: ~{(1-df[TARGET_COLUMN].mean())/df[TARGET_COLUMN].mean():.1f}')